# Data augmentation and regularization

**DeepVision Masterclass — Modern Edition**

This notebook is designed as an executable teaching lab. Read the dataset contract and validation notes before training.

## 1. Why augmentation?
Augmentation encodes plausible invariances. Transformations must preserve labels; for medical imagery, horizontal/vertical flips and rotations require domain-specific validation.

In [ ]:
import torch, torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
print(torch.__version__,torchvision.__version__)

## 2. Compare baseline and augmented pipelines
CIFAR-10 example. Do not apply random transformations to validation/test data.

In [ ]:
plain = transforms.Compose([transforms.ToTensor()])
aug = transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(p=.5),transforms.ColorJitter(.15,.15,.15,.05),transforms.ToTensor()])
ds_plain=datasets.CIFAR10('data',train=True,download=True,transform=plain)
ds_aug=datasets.CIFAR10('data',train=True,download=True,transform=aug)
labels=ds_plain.classes

## 3. Visualize stochastic views
The same source image can produce different training views.

In [ ]:
from PIL import Image
raw=ds_plain.data[12]
fig,ax=plt.subplots(1,5,figsize=(12,3))
ax[0].imshow(raw); ax[0].set_title('Original')
for i in range(1,5):
    t=aug(Image.fromarray(raw)); ax[i].imshow(t.permute(1,2,0)); ax[i].set_title(f'View {i}')
for a in ax:a.axis('off')

## 4. Regularization building blocks
Dropout, weight decay, label smoothing and mixup are complementary—not interchangeable.

In [ ]:
from torch import nn
loss_plain=nn.CrossEntropyLoss()
loss_smooth=nn.CrossEntropyLoss(label_smoothing=0.1)
# AdamW applies decoupled weight decay:
# optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
def mixup(x,y,alpha=.2):
    lam=np.random.beta(alpha,alpha)
    idx=torch.randperm(x.size(0),device=x.device)
    return lam*x+(1-lam)*x[idx],y,y[idx],lam
def mixup_loss(criterion,pred,ya,yb,lam):
    return lam*criterion(pred,ya)+(1-lam)*criterion(pred,yb)

## 5. TensorFlow/Keras equivalents
Use `tf.data` augmentation layers or preprocessing layers inside the model. They run only during training.

In [ ]:
import tensorflow as tf
print('TensorFlow:',tf.__version__)
keras_augment=tf.keras.Sequential([
 tf.keras.layers.RandomFlip('horizontal'),
 tf.keras.layers.RandomTranslation(.1,.1),
 tf.keras.layers.RandomRotation(.05),
 tf.keras.layers.RandomZoom(.1)])
# Example: inputs -> augmentation -> backbone. Keras preprocessing layers
# automatically respect training=True/False when called from a Model.

## 6. Experimental protocol
Run at least three seeds, keep an untouched test set, report mean±standard deviation, macro-F1, calibration and per-class recall. Record augmentation parameters and compute budget.

## Exercises
Compare augmentation policies; test label smoothing; implement mixup/CutMix; explain which transformations are invalid for digits, traffic signs, retinal fundus images and satellite imagery.